# 📊 Learn Data Visualization by Doing

**Who this is for:** you can write Python (variables, loops, functions, lists, dicts) but you've never worked with data before.

**What you'll do:** take a messy-ish dataset from raw table to clear, honest charts, following the same steps real analysts use:

1. **Load** the data
2. **Look** at it (what is in here?)
3. **Clean** it (fix what's broken)
4. **Ask a question**
5. **Pick a chart** that fits the question
6. **Build it**, then **make it readable**
7. **Share it**

## How this notebook works

| Symbol | Meaning |
|---|---|
| 🟦 **Worked example** | Code is already written. Read it, run it, change a number and see what happens. |
| ✏️ **Your turn** | Some code is missing. Replace each `___` with the right piece, then run the cell. |
| 💭 **Think** | A question to answer in your head (or in a new markdown cell). |

**Tips**
- Run a cell with **Shift + Enter**. Run cells **in order, top to bottom**.
- If you see `NameError: name '___' is not defined`, you haven't filled in a blank yet. That's expected!
- Every ✏️ cell has a `# HINT`. Stuck? Read the error message, it usually tells you what's wrong.
- Breaking things is the fastest way to learn. Change values and re-run.

## Step 0: Set up your tools

We'll use three libraries:

- **pandas**: holds data in tables (called *DataFrames*) and lets you slice and summarize them. Think of it as Excel inside Python.
- **matplotlib**: draws charts. It's the foundation most other plotting libraries are built on.
- **numpy**: fast math on lists of numbers. We'll barely touch it.

By convention people import them with short nicknames (`pd`, `plt`, `np`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Make charts show up inside the notebook and look a little nicer
%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 11

## Step 1: Load the data

Our dataset is a year (2024) of daily sales from a made-up coffee shop called **Byte & Brew**. Each row records how many cups of one drink were sold on one day.

Normally you'd load a file with `pd.read_csv("my_file.csv")`. So this notebook works anywhere with no downloads, the next cell *creates* the data instead.

> 🟦 **You don't need to understand the next cell.** It just builds the table. (If you're curious, it's a good place to practice reading code you didn't write.)

In [ ]:
# 🟦 Builds the practice dataset. No need to edit this.
rng = np.random.default_rng(42)
days = pd.date_range("2024-01-01", "2024-12-31")
temp = 55 + 25 * np.sin(2 * np.pi * (days.dayofyear - 110) / 366) + rng.normal(0, 6, len(days))

menu = {            # drink: (price, typical cups per day, how sales react to temperature)
    "Latte":       (5.00, 70, -0.35),
    "Espresso":    (3.50, 45, -0.05),
    "Iced Coffee": (4.50, 40,  0.90),
    "Hot Tea":     (3.00, 35, -0.30),
}

rows = []
for i, day in enumerate(days):
    weekend = day.dayofweek >= 5
    for drink, (price, base, slope) in menu.items():
        boost = (0.8 if drink == "Espresso" else 1.2) if weekend else 1.0
        cups = (base + slope * (temp[i] - 55)) * boost + rng.normal(0, 5)
        rows.append({
            "date": day.strftime("%Y-%m-%d"),   # stored as plain text on purpose!
            "drink": drink,
            "temp_f": round(temp[i], 1),
            "price": price,
            "cups_sold": int(max(cups, 0)),
        })

df = pd.DataFrame(rows)

# Real data is never perfect: a few missing temperatures and one typo
missing_days = rng.choice(days.strftime("%Y-%m-%d"), size=12, replace=False)
df.loc[df["date"].isin(missing_days), "temp_f"] = np.nan
df.loc[250, "cups_sold"] = 400

## Step 2: Look at your data

**Never chart data you haven't looked at.** Before drawing anything, answer:

- How big is it?
- What does each row mean? What does each column mean?
- What *type* is each column (numbers? text? dates?)
- Is anything missing or weird?

### ✏️ Your turn: four commands every analyst runs first

Fill in the blanks with these pandas methods:

| Goal | Code |
|---|---|
| Show the first few rows | `df.head()` |
| Rows × columns | `df.shape` (no parentheses!) |
| Column names, types, and non-empty counts | `df.info()` |
| Quick stats for number columns | `df.describe()` |

In [ ]:
# HINT: the method that shows the "head" (top) of a table
df.___()

In [ ]:
# HINT: this one is an attribute, not a method, so no parentheses
print("Rows and columns:", df.___)

In [ ]:
# HINT: the method that gives "info" about each column
df.___()

In [ ]:
# 🟦 Worked example
df.describe()

### 💭 Think

1. What does **one row** represent? (Write it as a sentence: "One row is ___.")
2. Look at the `info()` output. Which column has fewer non-null values than the others? That's a clue something is missing.
3. Look at the `date` column's type. It says `object`, which is pandas for "just text". Can we sort or group text by month? Not easily. We'll fix that next.

### ✏️ Your turn: explore one column

A single column is selected with square brackets: `df["drink"]`. Useful methods on a column:

- `.unique()`: the distinct values
- `.value_counts()`: how many times each value appears

In [ ]:
# HINT: which method lists the distinct values?
print(df["drink"].___())

# HINT: which method counts how often each value shows up?
df["drink"].___()

## Step 3: Clean the data

Cleaning isn't glamorous but it's usually where the most time goes. We have three jobs:

1. Turn the text dates into real dates
2. Deal with missing temperatures
3. Create new columns we'll need (revenue, month, weekday)

### 3a. Fix the date column

`pd.to_datetime()` converts text into proper dates. Once a column holds real dates, the `.dt` accessor unlocks things like `.dt.month` and `.dt.day_name()`.

In [ ]:
# HINT: the pandas function that converts text to datetime
df["date"] = pd.___(df["date"])

# 🟦 Now we can pull pieces out of the date
df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.day_name()

df[["date", "month", "weekday"]].head()

### 3b. Missing values

Missing values show up as `NaN` ("not a number"). First, count them per column. `df.isna()` gives True/False for every cell and `.sum()` counts the Trues.

In [ ]:
# 🟦 Worked example
df.isna().sum()

Twelve days have no temperature recorded (each day has 4 rows, one per drink, so that's 48 missing cells).

**What should we do?** There's no single right answer:

| Option | When it's reasonable |
|---|---|
| Drop those rows | You have lots of data and few missing |
| Fill with a typical value (median) | You need every row and the column isn't the star of your question |
| Leave them | The chart type ignores NaN anyway |

We'll fill with the **median** (the middle value, which isn't thrown off by extremes like the average can be). Just be aware that this is a choice, and in a real project you'd note it.

In [ ]:
# HINT: the method that fills in missing values; the Series method for the median
median_temp = df["temp_f"].___()
df["temp_f"] = df["temp_f"].___(median_temp)

print("Missing temperatures left:", df["temp_f"].isna().sum())

### 3c. Make a new column

Revenue = cups sold × price. Pandas does math on whole columns at once, so there's no loop needed.

In [ ]:
# HINT: multiply the two columns that make up revenue
df["revenue"] = df["cups_sold"] * df["___"]

df.head()

## Step 4: Start with a question, then pick a chart

The most common beginner mistake is to open a plotting library and try random charts. Instead:

> **Question first. Chart second.**

The *type* of question tells you which chart fits:

| Your question sounds like... | Use a... | Example |
|---|---|---|
| "What does the **distribution** of one number look like?" | **Histogram** | How many cups do we usually sell in a day? |
| "How do **categories compare**?" | **Bar chart** | Which drink earns the most? |
| "How does something **change over time**?" | **Line chart** | How did sales move month by month? |
| "Are **two numbers related**?" | **Scatter plot** | Do we sell more iced coffee when it's hotter? |
| "How **spread out** are values within groups?" | **Box plot** | Is Latte demand steady or erratic? |

We'll build one of each. Along the way you'll learn the anatomy of a matplotlib chart.

### The matplotlib pattern (memorize this)

```python
fig, ax = plt.subplots()      # 1. Make a blank canvas (fig) with one set of axes (ax)
ax.some_chart_method(data)    # 2. Draw something on the axes
ax.set_title("...")           # 3. Add labels so humans can read it
ax.set_xlabel("...")
ax.set_ylabel("...")
plt.show()                    # 4. Display it
```

`fig` is the whole picture. `ax` is the plot area inside it. Almost everything you do is calling methods on `ax`.

## Step 5: Histogram, "what does one number look like?"

A histogram chops a number line into **bins** and counts how many values land in each. Tall bar = common value.

### 🟦 Worked example: daily cups sold

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["cups_sold"], bins=30)
ax.set_title("How many cups do we sell per drink per day?")
ax.set_xlabel("Cups sold")
ax.set_ylabel("Number of days")
plt.show()

💭 **Think:** Most days cluster on the left, but there's a lonely bar way out on the right. Either we had one magical day, or someone made a typo.

### ✏️ Your turn: investigate the outlier

Filtering rows works like this: `df[df["column"] > 100]` keeps only rows where the condition is True.

In [ ]:
# HINT: look at the x-axis of the histogram. Pick a cutoff clearly above the main cluster.
suspicious = df[df["cups_sold"] > ___]
suspicious

One row claims 400 cups of a single drink in a day, while every other row is far lower (check with `df['cups_sold'].sort_values().tail()` if you like). That's almost certainly a typo (maybe 40?). We can't know for sure, so we'll **remove that row** and write down that we did.

The `~` symbol means "not". `df[~condition]` keeps everything that does *not* match.

In [ ]:
# HINT: keep rows where cups_sold is NOT over 200, i.e. use ~ with the same condition
df = df[~(df["cups_sold"] > ___)]
print("Rows remaining:", len(df))

### ✏️ Your turn: histogram of temperatures

Make a histogram of `temp_f` with 15 bins. Add a title and axis labels.

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["___"], bins=___)
ax.set_title("___")
ax.set_xlabel("___")
ax.set_ylabel("___")
plt.show()

💭 **Think:** Change `bins` to 3, then to 100. What happens to the shape? Too few bins hides detail; too many makes noise. There's no perfect number, so try a few.

## Step 6: Bar chart, "how do categories compare?"

Our question: **Which drink brings in the most revenue over the year?**

To answer it we need to **group** the data. `groupby` splits rows into piles by category, then you summarize each pile:

```python
df.groupby("drink")["revenue"].sum()
#  ^ make a pile per drink   ^ look at revenue  ^ add up each pile
```

In [ ]:
# HINT: which summary adds things up?
revenue_by_drink = df.groupby("drink")["revenue"].___()
revenue_by_drink = revenue_by_drink.sort_values()   # sorted bars are easier to read!
revenue_by_drink

Now plot it. For category names that can be long, **horizontal bars** (`barh`) are easier to read than vertical ones.

A pandas result like this has two parts: `.index` (the labels, here drink names) and `.values` (the numbers).

In [ ]:
fig, ax = plt.subplots()
ax.barh(revenue_by_drink.___, revenue_by_drink.___)
ax.set_title("Total 2024 revenue by drink")
ax.set_xlabel("___")
plt.show()

💭 **Think:** Revenue and cups sold tell different stories (because prices differ). Copy the cells above and change `"revenue"` to `"cups_sold"`. Does the ranking change?

## Step 7: Line chart, "how does it change over time?"

Lines connect points in order, which implies **one thing leading to the next**. That's why they're for time, not for unordered categories.

### 🟦 Worked example: total revenue per month

In [ ]:
monthly_revenue = df.groupby("month")["revenue"].sum()

fig, ax = plt.subplots()
ax.plot(monthly_revenue.index, monthly_revenue.values, marker="o")
ax.set_title("Revenue by month, 2024")
ax.set_xlabel("Month (1 = January)")
ax.set_ylabel("Revenue ($)")
ax.set_xticks(range(1, 13))
plt.show()

### ✏️ Your turn: one line per drink

Comparing several lines on the same chart is powerful, since you can see which drinks move together and which don't. The recipe:

1. Loop over each drink
2. Filter the data to just that drink
3. Group by month and total the cups
4. Draw one line, giving it a `label`
5. Call `ax.legend()` once at the end so the labels appear

In [ ]:
fig, ax = plt.subplots()

for drink in df["drink"].unique():
    subset = df[df["drink"] == ___]                  # HINT: the loop variable
    monthly = subset.groupby("month")["___"].sum()
    ax.plot(monthly.index, monthly.values, marker="o", label=___)

ax.set_title("Cups sold per month, by drink")
ax.set_xlabel("Month (1 = January)")
ax.set_ylabel("Cups sold")
ax.set_xticks(range(1, 13))
ax.___()                                            # HINT: shows the labels
plt.show()

💭 **Think:** Which drink peaks in summer? Which peak in winter? Why might that be?

## Step 8: Scatter plot, "are two numbers related?"

Each row becomes one dot: its **x** position is one number, its **y** position is another. If dots form a pattern (a slope, a curve), the two variables are related.

### 🟦 Worked example: iced coffee vs. temperature

In [ ]:
iced = df[df["drink"] == "Iced Coffee"]

fig, ax = plt.subplots()
ax.scatter(iced["temp_f"], iced["cups_sold"], alpha=0.5)
ax.set_title("Hotter days, more iced coffee")
ax.set_xlabel("Temperature (°F)")
ax.set_ylabel("Cups of iced coffee sold")
plt.show()

`alpha=0.5` makes dots half-transparent so overlapping dots darken, which is a handy trick when you have many points.

### ✏️ Your turn: compare two drinks

Plot **Iced Coffee** and **Latte** on the same axes in different colors. Do they react to temperature the same way?

In [ ]:
latte = df[df["drink"] == "___"]

fig, ax = plt.subplots()
ax.scatter(iced["temp_f"], iced["cups_sold"], alpha=0.5, color="tab:blue", label="Iced Coffee")
ax.scatter(latte["___"], latte["___"], alpha=0.5, color="tab:orange", label="Latte")
ax.set_title("How temperature affects sales")
ax.set_xlabel("Temperature (°F)")
ax.set_ylabel("Cups sold")
ax.legend()
plt.show()

> ⚠️ **Correlation is not causation.** The chart shows iced coffee and temperature move together. It doesn't *prove* heat causes the sales (though here it's a reasonable guess). Be careful with the words you use when you explain a scatter plot.

## Step 9: Box plot, "how spread out is each group?"

A box plot squeezes a distribution into five numbers:

- **Box**: the middle 50% of values (bottom edge = 25th percentile, top edge = 75th)
- **Line inside the box**: the median
- **Whiskers**: the typical range
- **Dots beyond**: unusually high or low values

It lets you compare many distributions side by side, which is awkward to do with histograms.

In [ ]:
drinks = list(df["drink"].unique())
cups_per_drink = [df[df["drink"] == d]["cups_sold"] for d in drinks]   # a list of 4 columns

fig, ax = plt.subplots()
ax.boxplot(cups_per_drink, tick_labels=___)         # HINT: the list of drink names we just made
ax.set_title("Daily cups sold, by drink")
ax.set_ylabel("Cups sold per day")
plt.show()

💭 **Think:** Which drink has the tallest box (most variable demand)? Which is the most predictable?

## Step 10: Make it clear and honest

A chart can be *technically correct* and still mislead. Look at this one:

### 🟦 A misleading chart

In [ ]:
avg_cups = df.groupby("drink")["cups_sold"].mean().sort_values()

fig, ax = plt.subplots()
ax.bar(avg_cups.index, avg_cups.values, color="red")
ax.set_ylim(30, 75)
plt.show()

**What's wrong?** List every problem you can find before reading on.

<details>
<summary>Click for the list</summary>

1. **No title or labels.** The reader has to guess what is being measured.
2. **Truncated y-axis.** The axis starts at 30 instead of 0. Bar *height* is supposed to represent *amount*, so cutting off the bottom makes small differences look huge.
3. **Distracting color.** Red usually means "bad/danger" and there's no reason for it here.
4. **Unrounded/unclear numbers.** Nothing tells you what the bars mean.
</details>

### ✏️ Your turn: fix it

Rewrite the chart so that it's honest and readable.

**The clarity checklist**
- ✅ A title that says what the chart *shows* (even better: what it *means*)
- ✅ Labeled axes **with units**
- ✅ Bars start at **zero**
- ✅ Sensible, calm colors
- ✅ Values labeled directly when there are only a few bars

In [ ]:
fig, ax = plt.subplots()
bars = ax.bar(avg_cups.index, avg_cups.values, color="steelblue")

ax.set_ylim(___, 80)                                    # HINT: where should bars start?
ax.set_title("___")   # HINT: say what the chart shows
ax.set_ylabel("___")
ax.bar_label(bars, fmt="%.0f")                          # 🟦 prints the number on each bar
plt.show()

## Step 11: Combine charts into a dashboard

`plt.subplots(rows, cols)` gives you a grid of axes. Instead of one `ax`, you get an array, so you pick one with `axes[row, col]`.

Counting starts at 0, just like list indexes: `axes[0, 0]` is top-left, `axes[1, 1]` is bottom-right.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

# Top-left: histogram (🟦 done for you)
axes[0, 0].hist(df["cups_sold"], bins=30, color="tab:gray")
axes[0, 0].set_title("Daily cups per drink")

# Top-right: horizontal bars (🟦 done for you)
axes[0, 1].barh(revenue_by_drink.index, revenue_by_drink.values, color="tab:green")
axes[0, 1].set_title("Total revenue by drink ($)")

# Bottom-left: line chart (✏️ fill in the position and data)
axes[___, ___].plot(monthly_revenue.index, monthly_revenue.values, marker="o", color="tab:purple")
axes[1, 0].set_title("Revenue by month ($)")
axes[1, 0].set_xticks(range(1, 13))

# Bottom-right: scatter (✏️ fill in the position and the columns)
axes[___, ___].scatter(iced["___"], iced["___"], alpha=0.4, color="tab:blue")
axes[1, 1].set_title("Iced coffee vs. temperature")

fig.suptitle("Byte & Brew: 2024 at a glance", fontsize=15)
fig.tight_layout()                                      # stops labels from overlapping
plt.show()

## Step 12: Save and share

`fig.savefig()` writes your chart to an image file you can drop into a slide, email, or report.

- `dpi=150` makes it sharp (higher = bigger file)
- `bbox_inches="tight"` trims extra whitespace

In [ ]:
fig.___("byte_and_brew_dashboard.png", dpi=150, bbox_inches="tight")
print("Saved! Look in the same folder as this notebook.")

## 🏁 Final challenge: ask your own question

Use everything above to answer **one question of your own** about the coffee shop. Some ideas:

- Are weekends busier than weekdays? *(hint: the `weekday` column, and `groupby`)*
- Which month was the best for Espresso?
- Does the price of a drink relate to how many cups we sell?
- What are the 10 best days by total revenue?

Follow the same workflow. Write your answers in the cells below.

**1. My question is:** *(double-click to edit)*

**2. The chart type that fits is:** *(histogram / bar / line / scatter / box, and why)*

**3. The data I need to prepare:** *(group? filter? new column?)*

In [ ]:
# Step A: prepare the data (filter / group / new column)

In [ ]:
# Step B: draw the chart (remember: fig, ax = plt.subplots() ... plt.show())

**4. What does the chart show?** Write one sentence a non-technical friend would understand.

**5. Self-check:**
- [ ] Does the chart have a title and labeled axes?
- [ ] Do bar charts start at zero?
- [ ] Would someone understand it without me explaining?
- [ ] Did I say anything the chart can't actually prove?

---

## Where to go next

| Skill | Try this |
|---|---|
| Prettier charts with less code | Learn **seaborn** (`import seaborn as sns`), built on matplotlib |
| Interactive charts (hover, zoom) | Try **plotly express** |
| Using your own data | Replace our data cell with `df = pd.read_csv("your_file.csv")`, then repeat Steps 2 to 3 |
| Finding datasets | Kaggle, data.gov, Our World in Data, TidyTuesday on GitHub |
| Better analysis | Learn `groupby` with multiple columns and `pivot_table` |

**The big idea:** *look at your data → ask a question → pick the chart that fits the question → make it clear.* The tools will change; that habit won't.